## **1. Get Face**
---

In [35]:
import os
import time
import cv2
import pickle

In [36]:
def face_tracking():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Error: Could not open webcam.')
        return

    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    while True:
        ret, frame = cap.read()
        frame      = cv2.flip(frame, 1)
        if not ret:
            print('Error capturing frame')
            break
        
        faces = detector.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.3, 5)
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        # time.sleep(0.05)
        
        cv2.imshow('Face Tracking (\'q\' to escape)', frame)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [37]:
def face_detector(pid, max_img=50):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Error: Could not open webcam.')
        return

    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    face_cnt = 0    
    while True:
        ret, frame = cap.read()
        frame      = cv2.flip(frame, 1)
        if not ret:
            print('Error capturing frame')
            break
                
        faces     = detector.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.3, 5)
        use_frame = frame.copy()
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
        if len(faces) > 0 and face_cnt < max_img:
            x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
            
            margin   = int(w * 0.1)
            x, y     = max(0, x - margin), max(0, y - margin)
            w, h     = min(use_frame.shape[1] - x, w + 2 * margin), min(use_frame.shape[0] - y, h + 2 * margin)
            face_img = cv2.resize(use_frame[y:y+h, x:x+w], (128, 128))
            
            cv2.imwrite(f"data/registered/{pid}/{pid}_{face_cnt:03d}.jpg", face_img)
            face_cnt += 1
            
        time.sleep(0.05)
        cv2.imshow('Face Registration (\'q\' to escape)', frame)
        if (cv2.waitKey(1) & 0xFF == ord('q')) or (face_cnt >= max_img):
            break
    
    cap.release()
    cv2.destroyAllWindows()

In [38]:
def register(info_path):
    cur  = sorted(os.listdir('data/registered'))[-1] if os.listdir('data/registered') else '-1 (no ID)'
    pid  = input(f'Enter person ID (last ID: {cur}): ')
    name = input('Enter person name: ')
    
    if pid in os.listdir('data/registered'):
        print(f"ID {pid} already exists. Please use a different ID.")
        return
    else:
        pdir = os.path.join('data/registered', pid)
        os.makedirs(pdir, exist_ok=True)
        
        # if os.path.exists(info_path):
        #     with open(info_path, 'rb') as f:
        #         info = pickle.load(f)
        # else:
        #     info = []
        # info.append({'pid': pid, 'name': name})
        # with open(info_path, 'wb') as f:
        #     pickle.dump(info, f)
            
        face_detector(pid, max_img=50)
        
    return pid, name

## **2. Build Embedding**
---

In [39]:
import os
import numpy as np
from PIL import Image

import faiss
import pickle

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

In [40]:
class EmbeddingModel(nn.Module):
    def __init__(self, num_layers_to_unfreeze=10, backbone='efficientnet_b0'):
        super().__init__()

        self.base_model            = timm.create_model(backbone, pretrained=True) # Backbone
        self.base_model.classifier = nn.Identity()                                # Remove classifier head

        # Only unfreeze last `num_layers_to_unfreeze` layers
        for param in self.base_model.parameters():
            param.requires_grad = False
        for param in list(self.base_model.parameters())[-num_layers_to_unfreeze:]:
            param.requires_grad = True

        # MLP Head
        feat_dim = self.base_model.num_features
        self.embedding_head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.SiLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256)     , nn.SiLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128)     , nn.SiLU(), nn.BatchNorm1d(128),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        feat = self.base_model(x)           # Extract feature from backbone, feat.shape=(B, feat_dim)
        emb  = self.embedding_head(feat)    # Get through MLP, emb.shape=(B, 128)
        emb  = F.normalize(emb, p=2, dim=1) # L2-normalize
        return emb

In [41]:
def load_model(ckpt_path, backbone, num_layers_to_unfreeze, device):
    model = EmbeddingModel(num_layers_to_unfreeze, backbone)
    model.to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])
    model.eval()
    return model


def build(emb_dim, index_path, info_path):
    if os.path.exists(index_path):
        index = faiss.read_index(index_path)
    else:
        index = faiss.IndexFlatIP(emb_dim)
        faiss.write_index(index, index_path)
        
    if os.path.exists(info_path):
        info = pickle.load(open(info_path,'rb'))
    else:
        info = []
        pickle.dump(info, open(info_path,'wb'))

    return index, info

In [42]:
def get_embedding(model, img_input, transform_method, device):
    if isinstance(img_input, str):
        img = Image.open(img_input).convert('RGB')
    elif isinstance(img_input, Image.Image):
        img = img_input.convert('RGB')
        
    img = transform_method(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        emb = model(img).cpu().numpy()
    return emb.astype(np.float32)


def add_face(pid, name, transform_method, model, device, index_path, info_path):
    pid_embs = []
    for filename in os.listdir(f'data/registered/{pid}'):
        img_path = os.path.join(f'data/registered/{pid}', filename)
        pid_embs.append(get_embedding(model, img_path, transform_method, device))
    pid_embs = np.concatenate(pid_embs, axis=0)
    
    index = faiss.read_index(index_path)
    index.add(pid_embs)
    faiss.write_index(index, index_path)
    
    info = pickle.load(open(info_path, 'rb'))
    for _ in range(pid_embs.shape[0]):
        info.append({'pid': pid, 'name': name})
    pickle.dump(info, open(info_path, 'wb'))

    return index, info

## **3. Face Recognition**
---

In [43]:
import os
import pickle

import cv2
import faiss
from PIL import Image

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [44]:
os.makedirs('data/registered', exist_ok=True)
os.makedirs('data/embedding', exist_ok=True)

INDEX_PATH = 'data/embedding/face_index.index'
INFO_PATH  = 'data/embedding/face_info.pkl'
CKPT_PATH  = 'results/models/best_embedding_model.pth'
EMB_DIM    = 128
THRESHOLD  = 0.5

In [45]:
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
model       = load_model(CKPT_PATH, 'efficientnet_b1', 20, device)
index, info = build(EMB_DIM, INDEX_PATH, INFO_PATH)

print(f"Device: {device}")
# print(f"Model: {model}")
print(f"Index size: {index.ntotal}")
print(f"Info size: {len(info)}")

Device: cpu
Index size: 0
Info size: 0


In [46]:
default_transform = transforms.Compose([
    transforms.Resize(150),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

In [47]:
# # 1. Test camera
# face_tracking()

In [52]:
# 2. Register face
pid, name = register(INFO_PATH)
add_face(pid, name, default_transform, model, device, INDEX_PATH, INFO_PATH)

(<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001A8625A4B70> >,
 [{'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name': 'Tai'},
  {'pid': '0', 'name':

In [49]:
def rebuild_index_and_info(index_path, info_path):
    index = faiss.read_index(index_path)
    info  = pickle.load(open(info_path,'rb'))
    return index, info

In [53]:
index, info = rebuild_index_and_info(INDEX_PATH, INFO_PATH)

print(f"Index size: {index.ntotal}")
print(f"Info size: {len(info)}")

Index size: 100
Info size: 100


In [54]:
# 3. FAISS search
def face_recognizer(index_path, info_path):
    index = faiss.read_index(index_path)
    info  = pickle.load(open(info_path,'rb'))
        
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Error: Could not open webcam.')
        return

    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    while True:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        if not ret:
            print('Error capturing frame')
            break

        faces = detector.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.3, 5)
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
            # Extract face, add margin
            margin             = int(w * 0.1)
            x_margin, y_margin = max(0, x-margin), max(0, y-margin)
            w_margin, h_margin = min(frame.shape[1]-x_margin, w+2*margin), min(frame.shape[0]-y_margin, h+2*margin)
            face_img           = frame[y_margin:y_margin+h_margin, x_margin:x_margin+w_margin]
            
            if face_img.size == 0:
                continue
                
            # Resize and convert to PIL Image
            face_img = cv2.resize(face_img, (128, 128))
            face_pil = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
            
            # Get embedding and search in the index
            emb  = get_embedding(model, face_pil, default_transform, device)
            D, I = index.search(emb, 1)
            
            # Display name if confidence is above threshold
            score = D[0][0]
            idx   = I[0][0]
            if score > THRESHOLD and idx < len(info):
                name = info[idx]['name']
                cv2.putText(frame, f"{name} - {score:.2f} - {idx}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            else:
                cv2.putText(frame, f"Unknown - {score:.2f} - {idx}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
        
        # time.sleep(0.05)
        cv2.imshow('Face Recognition (\'q\' to escape)', frame)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    
face_recognizer(INDEX_PATH, INFO_PATH)